In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

df = pd.read_csv("first_25000_rows.csv")

df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df.sort_values(by=['symbol', 'ts_event']).reset_index(drop=True)

In [3]:
# ------ 1. Best-Level OFI ------
df['dp_bid'] = df['bid_px_00'].diff()
df['ds_bid'] = df['bid_sz_00'].diff()
df['dp_ask'] = df['ask_px_00'].diff()
df['ds_ask'] = df['ask_sz_00'].diff()

def compute_best_ofi(row):
    ofi = 0
    if row['dp_bid'] > 0:
        ofi += row['bid_sz_00']
    elif row['dp_bid'] == 0:
        ofi += row['ds_bid']
    else:
        ofi -= row['bid_sz_00']
    if row['dp_ask'] < 0:
        ofi -= row['ask_sz_00']
    elif row['dp_ask'] == 0:
        ofi -= row['ds_ask']
    else:
        ofi += row['ask_sz_00']
    return ofi

df['ofi_best_level'] = df.apply(compute_best_ofi, axis=1)

In [4]:
# ------ 2. Multi-Level OFI ------
for level in range(10):
    bid_px = f'bid_px_0{level}'
    ask_px = f'ask_px_0{level}'
    bid_sz = f'bid_sz_0{level}'
    ask_sz = f'ask_sz_0{level}'

    dp_bid = df[bid_px].diff()
    ds_bid = df[bid_sz].diff()
    dp_ask = df[ask_px].diff()
    ds_ask = df[ask_sz].diff()

    level_ofi = []
    for i in range(len(df)):
        b_press = s_press = 0
        if pd.notna(dp_bid[i]):
            if dp_bid[i] > 0:
                b_press = df[bid_sz][i]
            elif dp_bid[i] == 0:
                b_press = ds_bid[i]
            else:
                b_press = -df[bid_sz][i]
            if dp_ask[i] < 0:
                s_press = df[ask_sz][i]
            elif dp_ask[i] == 0:
                s_press = ds_ask[i]
            else:
                s_press = -df[ask_sz][i]
        level_ofi.append(b_press - s_press)

    df[f'ofi_level_{level+1}'] = level_ofi

In [8]:
# ------ 3. Integrated OFI via PCA ------
ofi_cols = [f'ofi_level_{i}' for i in range(1, 11)]
ofi_matrix = df[ofi_cols].fillna(0).values
scaler = StandardScaler()
ofi_scaled = scaler.fit_transform(ofi_matrix)
pca = PCA(n_components=1)
ofi_pc1 = pca.fit_transform(ofi_scaled)
weights = np.abs(pca.components_[0])
weights /= weights.sum()
df['ofi_integrated'] = ofi_scaled @ weights

In [9]:
# ------ 4. Cross-Asset OFI (if multiple stocks exist within the dataset) ------
df['minute'] = df['ts_event'].dt.floor('min')
df['mid_price'] = (df['bid_px_00'] + df['ask_px_00']) / 2
df_minute = df.groupby(['symbol', 'minute']).last().reset_index()
df_minute['log_price'] = np.log(df_minute['mid_price'])
df_minute['log_return'] = df_minute.groupby('symbol')['log_price'].diff()

ofi_wide = df_minute.pivot(index='minute', columns='symbol', values='ofi_integrated')
ret_wide = df_minute.pivot(index='minute', columns='symbol', values='log_return')
panel_df = pd.concat([ofi_wide.add_suffix('_ofi'), ret_wide.add_suffix('_ret')], axis=1).dropna()

target_symbol = 'AAPL'
y = panel_df[f'{target_symbol}_ret'].values
X_cols = [col for col in panel_df.columns if col.endswith('_ofi') and col != f'{target_symbol}_ofi']
X = panel_df[X_cols].values

if X.shape[1] > 0:
    lasso = LassoCV(cv=5)
    lasso.fit(X, y)
    coef_dict = dict(zip(X_cols, lasso.coef_))
    sorted_coefs = sorted(coef_dict.items(), key=lambda x: -abs(x[1]))

    print(f"\nTop cross-impact contributors to {target_symbol}'s return:")
    for name, coef in sorted_coefs[:10]:
        print(f"{name}: {coef:.6f}")
else:
    print("Insufficient data for cross-asset OFI regression (only one asset present).")

Insufficient data for cross-asset OFI regression (only one asset present).
